<a href="https://colab.research.google.com/github/luiscano2330-bot/curso-programacion-cientifica/blob/main/taller_evaluativo_02_funciones_y_poo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Taller evaluativo 02 — Funciones y POO


## Ejercicio 1 — Calibrar una lectura

Defino `calibrar_lectura`, que aplica la recta de calibración `valor * pendiente + intercepto`. Los parámetros de calibración son opcionales, así que sin argumentos la lectura no cambia. Devuelve el resultado con `return`, no lo imprime.

In [1]:
def calibrar_lectura(valor, pendiente=1.0, intercepto=0.0):
    """Devuelve el valor calibrado: valor * pendiente + intercepto."""
    return valor * pendiente + intercepto

sin_calibrar = calibrar_lectura(10.0)
calibrada = calibrar_lectura(10.0, 1.2, -0.5)
con_intercepto = calibrar_lectura(10.0, intercepto=2.0)

print("Sin calibrar:", sin_calibrar)
print("Calibrada:", calibrada)
print("Con intercepto:", con_intercepto)

Sin calibrar: 10.0
Calibrada: 11.5
Con intercepto: 12.0


## Ejercicio 2 — ¿Es utilizable la lectura?

Una lectura sirve si no es el centinela -999.0 y está entre 0 y 500.0 inclusive. Defino `es_lectura_valida`, que recibe el valor por parámetro y devuelve un booleano, sin leer ninguna variable global.

In [2]:
lecturas_sbc_a = [10.0, -999.0, 25.0, 40.0, -999.0, 8.0, 52.0, 600.0]

def es_lectura_valida(valor):
    """Devuelve True si el valor no es el centinela y está entre 0 y 500."""
    return valor != -999.0 and 0.0 <= valor <= 500.0

print("25.0:", es_lectura_valida(25.0))
print("-999.0:", es_lectura_valida(-999.0))
print("600.0:", es_lectura_valida(600.0))
print("0.0:", es_lectura_valida(0.0))

25.0: True
-999.0: False
600.0: False
0.0: True


## Ejercicio 3 — Promedio de las lecturas útiles

`promedio_valido` recorre la lista que recibe, acumula solo las lecturas que pasan `es_lectura_valida` y devuelve el promedio. Si ninguna es válida, devuelve `None` en lugar de dividir entre cero.

In [3]:
def promedio_valido(lecturas):
    """Devuelve el promedio de las lecturas válidas o None si no hay ninguna."""
    suma = 0.0
    contador = 0
    for valor in lecturas:
        if es_lectura_valida(valor):
            suma += valor
            contador += 1
    if contador == 0:
        return None
    return suma / contador

promedio_sbc_a = promedio_valido(lecturas_sbc_a)

print("Promedio de SBC-A:", promedio_sbc_a)
print("Promedio sin lecturas válidas:", promedio_valido([-999.0, -999.0]))

Promedio de SBC-A: 27.0
Promedio sin lecturas válidas: None


## Ejercicio 4 — Calibrar la serie con map, filter y lambda

Con `filter` dejo solo las lecturas útiles de SBC-A, con `map` y una `lambda` las calibro con la pendiente y el intercepto del sensor, y con `filter` y otra `lambda` cuento cuántas superan 35.4. Sin `for` ni comprensiones de listas.

In [4]:
pendiente_sbc_a = 1.2
intercepto_sbc_a = -0.5

lecturas_utiles = filter(es_lectura_valida, lecturas_sbc_a)
lecturas_calibradas = list(map(lambda v: calibrar_lectura(v, pendiente_sbc_a, intercepto_sbc_a), lecturas_utiles))
excedencias = len(list(filter(lambda v: v > 35.4, lecturas_calibradas)))

print("Lecturas calibradas:", lecturas_calibradas)
print("Excedencias sobre 35.4:", excedencias)

Lecturas calibradas: [11.5, 29.5, 47.5, 9.1, 61.9]
Excedencias sobre 35.4: 2


## Ejercicio 5 — La clase Lectura

Convierto una lectura suelta en un objeto que sabe convertirse a ug/m3 y reconocer el centinela. Aquí la clase no valida: -999.0 es un valor legítimo que se puede guardar.

In [5]:
class Lectura:
    """Representa una lectura de PM2.5 con su valor, unidad y sensor."""
    def __init__(self, valor, unidad="ug/m3", codigo_sensor="sin asignar"):
        """Guarda el valor, la unidad y el código del sensor."""
        self.valor = valor
        self.unidad = unidad
        self.codigo_sensor = codigo_sensor

    def a_microgramos(self):
        """Devuelve la concentración convertida a ug/m3."""
        if self.unidad == "mg/m3":
            return self.valor * 1000
        return self.valor

    def es_dato_faltante(self):
        """Devuelve True si el valor es el centinela de dato faltante."""
        return self.valor == -999.0

lectura_campo = Lectura(0.0252, "mg/m3", "SBC-A")
lectura_faltante = Lectura(-999.0, codigo_sensor="SBC-A")
lectura_normal = Lectura(30.0)

print("Lectura de campo en ug/m3:", lectura_campo.a_microgramos())
print("¿La faltante es centinela?", lectura_faltante.es_dato_faltante())
print("¿La de campo es centinela?", lectura_campo.es_dato_faltante())
print("Unidad y sensor de la normal:", lectura_normal.unidad, lectura_normal.codigo_sensor)

Lectura de campo en ug/m3: 25.2
¿La faltante es centinela? True
¿La de campo es centinela? False
Unidad y sensor de la normal: ug/m3 sin asignar


## Ejercicio 6 — La clase Sensor: validación y encapsulamiento

La clase `Sensor` impide que exista con un código vacío o una pendiente no positiva: lanza `ValueError` antes de guardar nada. El código queda público y la pendiente e intercepto internos con guion bajo. `__repr__` y `__str__` hacen que el objeto se describa solo.

In [6]:
class Sensor:
    """Representa un sensor con su recta de calibración."""
    def __init__(self, codigo, pendiente=1.0, intercepto=0.0):
        """Crea el sensor validando el código y la pendiente."""
        if codigo == "":
            raise ValueError("El código del sensor no puede estar vacío")
        if pendiente <= 0:
            raise ValueError("La pendiente debe ser mayor que cero")
        self.codigo = codigo
        self._pendiente = pendiente
        self._intercepto = intercepto

    def calibrar(self, valor):
        """Devuelve la lectura calibrada con la recta del sensor."""
        return calibrar_lectura(valor, self._pendiente, self._intercepto)

    def __repr__(self):
        """Devuelve la representación completa del sensor."""
        return f"Sensor(codigo='{self.codigo}', pendiente={self._pendiente}, intercepto={self._intercepto})"

    def __str__(self):
        """Devuelve una descripción legible del sensor."""
        return f"Sensor {self.codigo}: pendiente {self._pendiente}, intercepto {self._intercepto}"

sensor_a = Sensor("SBC-A", 1.2, -0.5)
sensor_b = Sensor("SBC-B")

print(sensor_a)
print([sensor_a, sensor_b])
print("Calibración con SBC-A:", sensor_a.calibrar(10.0))
print("Calibración con SBC-B:", sensor_b.calibrar(10.0))

Sensor SBC-A: pendiente 1.2, intercepto -0.5
[Sensor(codigo='SBC-A', pendiente=1.2, intercepto=-0.5), Sensor(codigo='SBC-B', pendiente=1.0, intercepto=0.0)]
Calibración con SBC-A: 11.5
Calibración con SBC-B: 10.0


In [7]:
# Pruebas de la validación (dejadas comentadas para que el notebook no falle):
# Sensor("")        # -> ValueError: El código del sensor no puede estar vacío
# Sensor("X", 0)    # -> ValueError: La pendiente debe ser mayor que cero

## Ejercicio 7 — La clase Estacion: objetos que contienen objetos

La estación guarda sus lecturas como objetos `Lectura` en una lista interna. `valores()` obtiene la concentración en ug/m3 llamando a `a_microgramos()` de cada lectura, sin acceder a sus atributos directamente.

In [8]:
class Estacion:
    """Representa una estación con una lista interna de lecturas."""
    def __init__(self, codigo, sector):
        """Crea la estación con su código y sector, sin lecturas."""
        self.codigo = codigo
        self.sector = sector
        self._lecturas = []

    def agregar(self, lectura):
        """Agrega una Lectura a la estación."""
        self._lecturas.append(lectura)

    def valores(self):
        """Devuelve la concentración en ug/m3 de cada lectura guardada."""
        return [lectura.a_microgramos() for lectura in self._lecturas]

    def __repr__(self):
        """Devuelve la representación de la estación con su número de lecturas."""
        return f"Estacion('{self.codigo}', '{self.sector}', lecturas={len(self._lecturas)})"

datos_campana = [
    (10.0, "ug/m3"), (-999.0, "ug/m3"), (0.025, "mg/m3"), (40.0, "ug/m3"),
    (-999.0, "ug/m3"), (8.0, "ug/m3"), (52.0, "ug/m3"), (600.0, "ug/m3"),
]

estacion_oriente = Estacion("EST-04", "Oriente")
for valor, unidad in datos_campana:
    estacion_oriente.agregar(Lectura(valor, unidad, "SBC-A"))

print(estacion_oriente)
print(estacion_oriente.valores())

Estacion('EST-04', 'Oriente', lecturas=8)
[10.0, -999.0, 25.0, 40.0, -999.0, 8.0, 52.0, 600.0]


## Ejercicio 8 — Integración: el informe de la estación

Junto funciones y objetos: `InformeEstacion` calcula el promedio de las lecturas válidas ya calibradas reutilizando `es_lectura_valida`, `promedio_valido` y el método `calibrar` del sensor, y clasifica el resultado con `categoria_pm25`.

In [9]:
def categoria_pm25(concentracion):
    """Devuelve la categoría de la escala oficial para una concentración."""
    if concentracion <= 12.0:
        return "Buena"
    elif concentracion <= 35.4:
        return "Moderada"
    elif concentracion <= 55.4:
        return "Dañina para grupos sensibles"
    else:
        return "Dañina para la salud"

class InformeEstacion:
    """Genera el informe de una estación con un sensor."""
    def __init__(self, estacion, sensor):
        """Guarda la estación y el sensor del informe."""
        self._estacion = estacion
        self._sensor = sensor

    def promedio_calibrado(self):
        """Devuelve el promedio de las lecturas válidas ya calibradas."""
        valores = self._estacion.valores()
        utiles = filter(es_lectura_valida, valores)
        calibradas = list(map(lambda v: self._sensor.calibrar(v), utiles))
        return promedio_valido(calibradas)

    def categoria(self):
        """Devuelve la categoría del promedio calibrado."""
        return categoria_pm25(self.promedio_calibrado())

    def excedencias(self, umbral=35.4):
        """Devuelve cuántas lecturas calibradas superan el umbral."""
        valores = self._estacion.valores()
        utiles = filter(es_lectura_valida, valores)
        calibradas = list(map(lambda v: self._sensor.calibrar(v), utiles))
        return len(list(filter(lambda v: v > umbral, calibradas)))

    def __str__(self):
        """Devuelve el resumen legible del informe."""
        return f"{self._estacion.codigo}: {round(self.promedio_calibrado(), 1)} ug/m3 - {self.categoria()}"

informe = InformeEstacion(estacion_oriente, sensor_a)

print(informe)
print("Promedio calibrado:", informe.promedio_calibrado())

EST-04: 31.9 ug/m3 - Moderada
Promedio calibrado: 31.9


## Verificación final

Ejecuto la celda de verificación del taller para comprobar funciones, objetos y algunos resultados intermedios.

In [10]:
# Celda de verificación — ejecútela al final, después de todo lo demás
assert abs(calibrar_lectura(10.0, 1.2, -0.5) - 11.5) < 1e-9, "revise el Ejercicio 1"
assert calibrar_lectura(10.0) == 10.0, "sin parámetros no debe alterar la lectura"
assert abs(con_intercepto - 12.0) < 1e-9, "revise con_intercepto del Ejercicio 1"
assert es_lectura_valida(25.0) and not es_lectura_valida(-999.0), "revise el Ejercicio 2"
assert not es_lectura_valida(600.0), "600.0 excede el rango del sensor"
assert abs(promedio_sbc_a - 27.0) < 1e-9, "revise el promedio del Ejercicio 3"
assert promedio_valido([-999.0, -999.0]) is None, "sin lecturas válidas debe devolver None"
assert len(lecturas_calibradas) == 5, "deberían quedar 5 lecturas calibradas"
assert abs(lecturas_calibradas[0] - 11.5) < 1e-9, "revise la calibración del Ejercicio 4"
assert excedencias == 2, "deberían ser 2 lecturas sobre 35.4"
assert abs(lectura_campo.a_microgramos() - 25.2) < 1e-9, "revise a_microgramos"
assert lectura_faltante.es_dato_faltante() and not lectura_campo.es_dato_faltante()
assert repr(sensor_a) == "Sensor(codigo='SBC-A', pendiente=1.2, intercepto=-0.5)", "revise __repr__ de Sensor"
assert hasattr(sensor_a, "_pendiente") and not hasattr(sensor_a, "pendiente"), "la pendiente debe ser interna"
assert abs(sensor_b.calibrar(10.0) - 10.0) < 1e-9, "revise los valores por defecto de Sensor"
assert repr(estacion_oriente) == "Estacion('EST-04', 'Oriente', lecturas=8)", "revise __repr__ de Estacion"
assert estacion_oriente.valores() == lecturas_sbc_a, "revise valores() del Ejercicio 7"
assert abs(informe.promedio_calibrado() - 31.9) < 1e-9, "revise el Ejercicio 8"
assert informe.categoria() == "Moderada", "revise categoria()"
assert str(informe) == "EST-04: 31.9 ug/m3 - Moderada", "revise __str__ de InformeEstacion"
print("Verificación completada sin errores.")

Verificación completada sin errores.
